# Transformer baseline for injection-bench (Colab)

Scores a pretrained Hugging Face prompt-injection classifier on the same time-split JailbreakHub test set as the other detectors, and writes a report you can paste into the README and paper.

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU** (CPU also works but takes much longer).
2. Run the cells in order. Total time is roughly 5-15 minutes on a T4.
3. For *gated* models (for example `meta-llama/Prompt-Guard-86M`): accept the licence on the model's Hugging Face page, create a read token, and add it in Colab under the key icon (Secrets) as `HF_TOKEN`. **Never paste a token into a cell or into chat.**

**Read before you quote the numbers.** Pretrained detectors were not trained on this notebook's training split, so the train/test time split does not apply to them, and their training data may include prompts from this public dataset. Report them as *"pretrained, possible overlap with training data"* and do not claim they beat the other rows.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Install

In [ ]:
!pip install -q "injection-bench[hf,sklearn] @ git+https://github.com/ashishdhillon/injection-bench.git" guardkit
import injectbench; print("injectbench", injectbench.__version__)

## 2. Hugging Face token (only needed for gated models)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as e:
    print("No HF_TOKEN secret found (fine for public models):", type(e).__name__)

## 3. Download the public dataset (Shen et al., CCS 2024, MIT licence)

In [ ]:
!injectbench download data/
!ls -la data/

## 4. Pick the transformer model(s)

Edit `MODELS`. Suggested public starting point: `protectai/deberta-v3-base-prompt-injection-v2`. Check each model card for its licence, labels and training data (I could not verify the model IDs from the sandbox where this was written). If a model's labels are ambiguous the run stops with a clear error; then name the attack label as `hf:MODEL_ID#LABEL`.

In [ ]:
MODELS = [
    "hf:protectai/deberta-v3-base-prompt-injection-v2",
    # "hf:meta-llama/Prompt-Guard-86M",     # gated: needs HF_TOKEN and accepted licence
]
BASELINES = ["length", "keywords", "hashed-lr", "tfidf-lr", "guardkit:rules", "guardkit:model"]
args = " ".join(f"--detector {d}" for d in BASELINES + MODELS)
print(args)

## 5. Run (writes report.md and report.json)

In [ ]:
!injectbench run --data data/ {args} --boot 200 --out report.md

## 6. Show and save the results

In [ ]:
print(open("report.md").read())

In [ ]:
from google.colab import files
files.download("report.md"); files.download("report.json")

## Troubleshooting
- **`cannot tell which label means 'attack'`**: open the model card, then use `hf:MODEL_ID#LABEL_NAME`.
- **Out of memory**: Runtime -> Restart, then re-run; or edit `batch_size` (default 16) in `HFClassifier`.
- **401 / 403 for a gated model**: the licence is not accepted for the token's account, or `HF_TOKEN` is missing.
- **Very slow**: confirm the GPU line in the first cell prints `True`.

When done, send `report.md` to the assistant (or open a pull request) to add the rows to the README and the paper, with the caveat above.